# E-Waste Component Classification
**SDG 12.4 | Predictive Analysis Project**  
Transfer learning comparison: ResNet18 vs ResNet50 vs EfficientNet-B0

## environment

In [1]:
# verify gpu availability before proceeding
import torch
print(f"pytorch     : {torch.__version__}")
print(f"cuda        : {torch.cuda.is_available()}")
print(f"gpu         : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none'}")
print(f"vram        : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB" if torch.cuda.is_available() else "")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device      : {device}")

pytorch     : 2.11.0+cu126
cuda        : True
gpu         : NVIDIA GeForce RTX 3050 6GB Laptop GPU
vram        : 6.4 GB
device      : cuda


## imports

In [2]:
import os
import time
import json
import warnings
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from copy import deepcopy

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, WeightedRandomSampler
from torchvision import datasets, transforms, models
from torch.cuda.amp import GradScaler, autocast

from sklearn.metrics import (
    classification_report, confusion_matrix,
    f1_score, accuracy_score, precision_score, recall_score
)
from tqdm import tqdm

warnings.filterwarnings("ignore")
torch.manual_seed(42)
np.random.seed(42)
torch.backends.cudnn.benchmark = True

## configuration

In [3]:
# paths
DATA_DIR   = Path(r"D:\Downloads\MASTER_DATASET")
OUTPUT_DIR = Path(r"D:\Github Desktop\ewaste_vit_project\models\classification")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
GRAPHS_DIR = OUTPUT_DIR / "graphs"
GRAPHS_DIR.mkdir(exist_ok=True)

# hyperparameters — tuned for rtx 3050 4gb vram
CONFIG = {
    "img_size"    : 224,
    "batch_size"  : 32,       # safe for 3050 with amp enabled
    "num_epochs"  : 30,
    "lr"          : 1e-4,
    "weight_decay": 1e-4,
    "num_classes" : 20,
    "num_workers" : 4,
    "patience"    : 7,        # early stopping patience
    "unfreeze_epoch": 5,      # epoch to start fine-tuning backbone
}

print("configuration loaded")
print(json.dumps({k: str(v) for k, v in CONFIG.items()}, indent=2))

configuration loaded
{
  "img_size": "224",
  "batch_size": "32",
  "num_epochs": "30",
  "lr": "0.0001",
  "weight_decay": "0.0001",
  "num_classes": "20",
  "num_workers": "4",
  "patience": "7",
  "unfreeze_epoch": "5"
}


## data pipeline

In [4]:
MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]

# training transforms — aggressive augmentation for robustness
train_transform = transforms.Compose([
    transforms.Resize((CONFIG["img_size"] + 32, CONFIG["img_size"] + 32)),
    transforms.RandomCrop(CONFIG["img_size"]),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.05),
    transforms.RandomGrayscale(p=0.05),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
    transforms.RandomErasing(p=0.1, scale=(0.02, 0.1)),  # simulate partial occlusion
])

# val/test transforms — no augmentation, deterministic
eval_transform = transforms.Compose([
    transforms.Resize((CONFIG["img_size"], CONFIG["img_size"])),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

# load datasets
train_ds = datasets.ImageFolder(DATA_DIR / "train", transform=train_transform)
val_ds   = datasets.ImageFolder(DATA_DIR / "val",   transform=eval_transform)
test_ds  = datasets.ImageFolder(DATA_DIR / "test",  transform=eval_transform)

CLASS_NAMES = train_ds.classes
NUM_CLASSES = len(CLASS_NAMES)

print(f"classes     : {NUM_CLASSES}")
print(f"train imgs  : {len(train_ds)}")
print(f"val imgs    : {len(val_ds)}")
print(f"test imgs   : {len(test_ds)}")
print(f"class list  : {CLASS_NAMES}")

classes     : 20
train imgs  : 20273
val imgs    : 2000
test imgs   : 2000
class list  : ['Battery', 'Capacitor', 'Integrated-micro-circuit', 'Keyboard', 'LED', 'Laptop', 'Microwave', 'Mobile', 'Mouse', 'PCB', 'Printer', 'Resistor', 'Television', 'Washing Machine', 'heat-sink', 'light bulbs', 'microchip', 'microprocessor', 'semiconductor-diode', 'transistor']


## weighted sampler — handles class imbalance

In [5]:
# compute per-sample weights inversely proportional to class frequency
# ensures underrepresented classes get equal training exposure
class_counts  = np.bincount([s[1] for s in train_ds.samples])
class_weights = 1.0 / class_counts
sample_weights = np.array([class_weights[s[1]] for s in train_ds.samples])

sampler = WeightedRandomSampler(
    weights     = torch.DoubleTensor(sample_weights),
    num_samples = len(sample_weights),
    replacement = True
)

train_loader = DataLoader(
    train_ds, batch_size=CONFIG["batch_size"],
    sampler=sampler, num_workers=CONFIG["num_workers"],
    pin_memory=True
)
val_loader = DataLoader(
    val_ds, batch_size=CONFIG["batch_size"],
    shuffle=False, num_workers=CONFIG["num_workers"],
    pin_memory=True
)
test_loader = DataLoader(
    test_ds, batch_size=CONFIG["batch_size"],
    shuffle=False, num_workers=CONFIG["num_workers"],
    pin_memory=True
)

print(f"train batches : {len(train_loader)}")
print(f"val batches   : {len(val_loader)}")
print(f"test batches  : {len(test_loader)}")
print("weighted sampler applied")

train batches : 634
val batches   : 63
test batches  : 63
weighted sampler applied


## model factory

In [6]:
def build_model(arch, num_classes):
    """build pretrained model with custom classification head."""

    if arch == "resnet18":
        model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
        in_features = model.fc.in_features
        model.fc = nn.Sequential(
            nn.Linear(in_features, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.4),
            nn.Linear(512, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )

    elif arch == "resnet50":
        model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
        in_features = model.fc.in_features
        model.fc = nn.Sequential(
            nn.Linear(in_features, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.4),
            nn.Linear(512, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )

    elif arch == "efficientnet_b0":
        model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
        in_features = model.classifier[1].in_features
        model.classifier = nn.Sequential(
            nn.Dropout(0.4),
            nn.Linear(in_features, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )

    else:
        raise ValueError(f"unknown architecture: {arch}")

    # freeze backbone initially — only train head
    for name, param in model.named_parameters():
        if "fc" not in name and "classifier" not in name:
            param.requires_grad = False

    return model


def count_params(model):
    total     = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable


# verify model builds correctly
test_model = build_model("resnet18", NUM_CLASSES)
total, trainable = count_params(test_model)
print(f"resnet18 total params     : {total:,}")
print(f"resnet18 trainable params : {trainable:,}")
del test_model

resnet18 total params     : 11,576,660
resnet18 trainable params : 400,148


## training engine

In [7]:
def unfreeze_backbone(model, arch, layer_name=None):
    """progressively unfreeze backbone layers for fine-tuning."""
    if arch in ["resnet18", "resnet50"]:
        for name, param in model.named_parameters():
            if "layer4" in name or "layer3" in name or "fc" in name:
                param.requires_grad = True
    elif arch == "efficientnet_b0":
        for name, param in model.named_parameters():
            if "features.7" in name or "features.8" in name or "classifier" in name:
                param.requires_grad = True
    return model


def train_epoch(model, loader, criterion, optimizer, scaler, device):
    model.train()
    total_loss = correct = total = 0

    for images, labels in loader:
        images, labels = images.to(device, non_blocking=True), labels.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)

        with autocast():
            outputs = model(images)
            loss    = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item() * images.size(0)
        _, preds    = torch.max(outputs, 1)
        correct    += (preds == labels).sum().item()
        total      += labels.size(0)

    return total_loss / total, correct / total


def eval_epoch(model, loader, criterion, device):
    model.eval()
    total_loss = correct = total = 0
    all_preds, all_labels = [], []

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device, non_blocking=True), labels.to(device, non_blocking=True)
            with autocast():
                outputs = model(images)
                loss    = criterion(outputs, labels)
            total_loss += loss.item() * images.size(0)
            _, preds    = torch.max(outputs, 1)
            correct    += (preds == labels).sum().item()
            total      += labels.size(0)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    return total_loss / total, correct / total, all_preds, all_labels


def train_model(arch, device, config, save_dir):
    """full training loop with early stopping and lr scheduling."""
    print(f"\ntraining {arch}")
    print("-" * 50)

    model     = build_model(arch, config["num_classes"]).to(device)
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    optimizer = optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=config["lr"], weight_decay=config["weight_decay"]
    )
    scheduler = optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=config["num_epochs"], eta_min=1e-6
    )
    scaler = GradScaler()

    history      = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": [], "lr": []}
    best_val_acc = 0.0
    best_weights = None
    patience_ctr = 0
    best_path    = save_dir / f"{arch}_best.pth"

    for epoch in range(1, config["num_epochs"] + 1):
        t0 = time.time()

        # unfreeze backbone at configured epoch
        if epoch == config["unfreeze_epoch"]:
            model = unfreeze_backbone(model, arch)
            optimizer = optim.AdamW(
                filter(lambda p: p.requires_grad, model.parameters()),
                lr=config["lr"] * 0.1, weight_decay=config["weight_decay"]
            )
            scheduler = optim.lr_scheduler.CosineAnnealingLR(
                optimizer, T_max=config["num_epochs"] - epoch, eta_min=1e-7
            )
            print(f"  epoch {epoch}: backbone unfrozen for fine-tuning")

        train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, scaler, device)
        val_loss, val_acc, _, _ = eval_epoch(model, val_loader, criterion, device)
        scheduler.step()

        current_lr = optimizer.param_groups[0]["lr"]
        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)
        history["lr"].append(current_lr)

        elapsed = time.time() - t0
        print(
            f"  epoch {epoch:02d}/{config['num_epochs']} | "
            f"loss {train_loss:.4f}/{val_loss:.4f} | "
            f"acc {train_acc:.4f}/{val_acc:.4f} | "
            f"lr {current_lr:.2e} | {elapsed:.1f}s"
        )

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_weights = deepcopy(model.state_dict())
            torch.save(model.state_dict(), best_path)
            patience_ctr = 0
        else:
            patience_ctr += 1
            if patience_ctr >= config["patience"]:
                print(f"  early stopping at epoch {epoch}")
                break

    model.load_state_dict(best_weights)
    print(f"  best val accuracy: {best_val_acc:.4f}")
    return model, history, best_val_acc


print("training engine ready")

training engine ready


## train all three architectures

In [8]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ARCHS  = ["resnet18", "resnet50", "efficientnet_b0"]

all_histories = {}
all_models    = {}

for arch in ARCHS:
    arch_dir = OUTPUT_DIR / arch
    arch_dir.mkdir(exist_ok=True)
    model, history, best_acc = train_model(arch, device, CONFIG, arch_dir)
    all_histories[arch] = history
    all_models[arch]    = model
    torch.cuda.empty_cache()

print("\nall models trained")


training resnet18
--------------------------------------------------
  epoch 01/30 | loss 2.2881/1.5480 | acc 0.4038/0.6530 | lr 9.97e-05 | 75.7s
  epoch 02/30 | loss 1.6966/1.3775 | acc 0.5850/0.7095 | lr 9.89e-05 | 89.7s
  epoch 03/30 | loss 1.5763/1.3486 | acc 0.6310/0.7215 | lr 9.76e-05 | 71.9s
  epoch 04/30 | loss 1.5191/1.3329 | acc 0.6519/0.7225 | lr 9.57e-05 | 70.1s
  epoch 5: backbone unfrozen for fine-tuning
  epoch 05/30 | loss 1.4112/1.2142 | acc 0.6935/0.7705 | lr 9.96e-06 | 79.3s
  epoch 06/30 | loss 1.2988/1.1657 | acc 0.7412/0.7865 | lr 9.84e-06 | 72.4s
  epoch 07/30 | loss 1.2327/1.1318 | acc 0.7635/0.8010 | lr 9.65e-06 | 70.0s
  epoch 08/30 | loss 1.1879/1.1203 | acc 0.7853/0.8045 | lr 9.39e-06 | 70.6s
  epoch 09/30 | loss 1.1479/1.0903 | acc 0.8027/0.8175 | lr 9.05e-06 | 71.5s
  epoch 10/30 | loss 1.1191/1.0782 | acc 0.8139/0.8205 | lr 8.66e-06 | 72.2s
  epoch 11/30 | loss 1.0975/1.0597 | acc 0.8205/0.8265 | lr 8.21e-06 | 71.6s
  epoch 12/30 | loss 1.0837/1.0402 | a

100%|██████████| 97.8M/97.8M [00:33<00:00, 3.03MB/s]


  epoch 01/30 | loss 1.9818/1.2717 | acc 0.5236/0.7485 | lr 9.97e-05 | 74.4s
  epoch 02/30 | loss 1.4214/1.1632 | acc 0.6910/0.7730 | lr 9.89e-05 | 74.8s
  epoch 03/30 | loss 1.3252/1.1439 | acc 0.7288/0.7855 | lr 9.76e-05 | 73.8s
  epoch 04/30 | loss 1.2732/1.1195 | acc 0.7468/0.7915 | lr 9.57e-05 | 74.5s
  epoch 5: backbone unfrozen for fine-tuning
  epoch 05/30 | loss 1.1991/1.0807 | acc 0.7780/0.8080 | lr 9.96e-06 | 103.7s
  epoch 06/30 | loss 1.1296/1.0550 | acc 0.8053/0.8160 | lr 9.84e-06 | 101.3s
  epoch 07/30 | loss 1.0889/1.0303 | acc 0.8212/0.8305 | lr 9.65e-06 | 101.4s
  epoch 08/30 | loss 1.0564/1.0038 | acc 0.8337/0.8300 | lr 9.39e-06 | 101.1s
  epoch 09/30 | loss 1.0182/0.9956 | acc 0.8525/0.8325 | lr 9.05e-06 | 101.2s
  epoch 10/30 | loss 1.0055/0.9951 | acc 0.8538/0.8380 | lr 8.66e-06 | 101.2s
  epoch 11/30 | loss 0.9821/0.9674 | acc 0.8651/0.8455 | lr 8.21e-06 | 101.0s
  epoch 12/30 | loss 0.9657/0.9752 | acc 0.8697/0.8430 | lr 7.70e-06 | 101.2s
  epoch 13/30 | loss 0.

100%|██████████| 20.5M/20.5M [01:26<00:00, 249kB/s] 


  epoch 01/30 | loss 2.2362/1.4552 | acc 0.4439/0.6810 | lr 9.97e-05 | 126.4s
  epoch 02/30 | loss 1.4935/1.2298 | acc 0.6628/0.7540 | lr 9.89e-05 | 81.0s
  epoch 03/30 | loss 1.3324/1.1864 | acc 0.7201/0.7715 | lr 9.76e-05 | 81.0s
  epoch 04/30 | loss 1.2470/1.1299 | acc 0.7529/0.7875 | lr 9.57e-05 | 81.2s
  epoch 5: backbone unfrozen for fine-tuning
  epoch 05/30 | loss 1.2035/1.0858 | acc 0.7715/0.7995 | lr 9.96e-06 | 88.0s
  epoch 06/30 | loss 1.1924/1.0760 | acc 0.7754/0.8080 | lr 9.84e-06 | 81.9s
  epoch 07/30 | loss 1.1864/1.1029 | acc 0.7775/0.7970 | lr 9.65e-06 | 83.0s
  epoch 08/30 | loss 1.1718/1.0673 | acc 0.7794/0.8105 | lr 9.39e-06 | 82.3s
  epoch 09/30 | loss 1.1585/1.0747 | acc 0.7838/0.8115 | lr 9.05e-06 | 81.9s
  epoch 10/30 | loss 1.1506/1.0686 | acc 0.7889/0.8110 | lr 8.66e-06 | 82.3s
  epoch 11/30 | loss 1.1389/1.0367 | acc 0.7948/0.8270 | lr 8.21e-06 | 82.9s
  epoch 12/30 | loss 1.1364/1.0338 | acc 0.7972/0.8280 | lr 7.70e-06 | 82.0s
  epoch 13/30 | loss 1.1222/1.

## plot training curves

In [9]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle("training curves — e-waste classification", fontsize=14, fontweight="bold")

for idx, arch in enumerate(ARCHS):
    h      = all_histories[arch]
    epochs = range(1, len(h["train_loss"]) + 1)

    # loss
    axes[0, idx].plot(epochs, h["train_loss"], label="train", color="#2c7bb6", linewidth=1.5)
    axes[0, idx].plot(epochs, h["val_loss"],   label="val",   color="#d7191c", linewidth=1.5)
    axes[0, idx].set_title(f"{arch} — loss", fontsize=11)
    axes[0, idx].set_xlabel("epoch")
    axes[0, idx].set_ylabel("cross-entropy loss")
    axes[0, idx].legend(fontsize=9)
    axes[0, idx].grid(alpha=0.3)

    # accuracy
    axes[1, idx].plot(epochs, h["train_acc"], label="train", color="#2c7bb6", linewidth=1.5)
    axes[1, idx].plot(epochs, h["val_acc"],   label="val",   color="#d7191c", linewidth=1.5)
    axes[1, idx].set_title(f"{arch} — accuracy", fontsize=11)
    axes[1, idx].set_xlabel("epoch")
    axes[1, idx].set_ylabel("accuracy")
    axes[1, idx].legend(fontsize=9)
    axes[1, idx].grid(alpha=0.3)
    axes[1, idx].set_ylim(0, 1)

plt.tight_layout()
save_path = GRAPHS_DIR / "training_curves.png"
plt.savefig(save_path, dpi=150, bbox_inches="tight")
plt.close()
print(f"saved: {save_path}")

saved: D:\Github Desktop\ewaste_vit_project\models\classification\graphs\training_curves.png


## evaluate on test set

In [10]:
test_results = {}

for arch in ARCHS:
    model     = all_models[arch]
    criterion = nn.CrossEntropyLoss()
    test_loss, test_acc, preds, labels = eval_epoch(model, test_loader, criterion, device)

    macro_f1  = f1_score(labels, preds, average="macro")
    weighted_f1 = f1_score(labels, preds, average="weighted")
    precision = precision_score(labels, preds, average="macro")
    recall    = recall_score(labels, preds, average="macro")

    test_results[arch] = {
        "test_accuracy"  : round(test_acc,     4),
        "test_loss"      : round(test_loss,     4),
        "macro_f1"       : round(macro_f1,      4),
        "weighted_f1"    : round(weighted_f1,   4),
        "macro_precision": round(precision,     4),
        "macro_recall"   : round(recall,        4),
        "preds"          : preds,
        "labels"         : labels,
    }

    print(f"\n{arch}")
    print(f"  accuracy        : {test_acc:.4f}")
    print(f"  macro f1        : {macro_f1:.4f}")
    print(f"  weighted f1     : {weighted_f1:.4f}")
    print(f"  macro precision : {precision:.4f}")
    print(f"  macro recall    : {recall:.4f}")

# save numeric results to json
json_results = {
    arch: {k: v for k, v in r.items() if k not in ["preds", "labels"]}
    for arch, r in test_results.items()
}
with open(OUTPUT_DIR / "test_results.json", "w") as f:
    json.dump(json_results, f, indent=2)
print("\nresults saved to test_results.json")


resnet18
  accuracy        : 0.8575
  macro f1        : 0.8543
  weighted f1     : 0.8543
  macro precision : 0.8569
  macro recall    : 0.8575

resnet50
  accuracy        : 0.8655
  macro f1        : 0.8619
  weighted f1     : 0.8619
  macro precision : 0.8659
  macro recall    : 0.8655

efficientnet_b0
  accuracy        : 0.8265
  macro f1        : 0.8207
  weighted f1     : 0.8207
  macro precision : 0.8310
  macro recall    : 0.8265

results saved to test_results.json


## confusion matrices — all three models

In [11]:
fig, axes = plt.subplots(1, 3, figsize=(28, 9))
fig.suptitle("confusion matrices — test set", fontsize=14, fontweight="bold")

for idx, arch in enumerate(ARCHS):
    cm = confusion_matrix(test_results[arch]["labels"], test_results[arch]["preds"])
    # normalize to percentage for readability
    cm_norm = cm.astype("float") / cm.sum(axis=1)[:, np.newaxis]

    sns.heatmap(
        cm_norm, ax=axes[idx],
        annot=True, fmt=".2f", cmap="Blues",
        xticklabels=CLASS_NAMES,
        yticklabels=CLASS_NAMES,
        linewidths=0.3, cbar=True,
        vmin=0, vmax=1
    )
    axes[idx].set_title(
        f"{arch}\nacc={test_results[arch]['test_accuracy']:.4f}  f1={test_results[arch]['macro_f1']:.4f}",
        fontsize=10
    )
    axes[idx].set_ylabel("true label", fontsize=9)
    axes[idx].set_xlabel("predicted label", fontsize=9)
    axes[idx].tick_params(axis="x", rotation=45, labelsize=7)
    axes[idx].tick_params(axis="y", rotation=0,  labelsize=7)

plt.tight_layout()
save_path = GRAPHS_DIR / "confusion_matrices.png"
plt.savefig(save_path, dpi=150, bbox_inches="tight")
plt.close()
print(f"saved: {save_path}")

saved: D:\Github Desktop\ewaste_vit_project\models\classification\graphs\confusion_matrices.png


## per-class f1 comparison

In [12]:
per_class_f1 = {}
for arch in ARCHS:
    per_class_f1[arch] = f1_score(
        test_results[arch]["labels"],
        test_results[arch]["preds"],
        average=None
    )

x      = np.arange(len(CLASS_NAMES))
width  = 0.25
colors = ["#2c7bb6", "#1a9641", "#d7191c"]

fig, ax = plt.subplots(figsize=(18, 7))
for i, (arch, color) in enumerate(zip(ARCHS, colors)):
    bars = ax.bar(x + i * width, per_class_f1[arch], width,
                  label=arch, color=color, alpha=0.85, edgecolor="white")

ax.set_title("per-class f1 score — model comparison", fontsize=13, fontweight="bold")
ax.set_xlabel("class", fontsize=11)
ax.set_ylabel("f1 score", fontsize=11)
ax.set_xticks(x + width)
ax.set_xticklabels(CLASS_NAMES, rotation=45, ha="right", fontsize=9)
ax.legend(fontsize=10)
ax.axhline(y=0.85, color="gray", linestyle="--", alpha=0.6, label="target f1=0.85")
ax.set_ylim(0, 1.05)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
save_path = GRAPHS_DIR / "per_class_f1_comparison.png"
plt.savefig(save_path, dpi=150, bbox_inches="tight")
plt.close()
print(f"saved: {save_path}")

saved: D:\Github Desktop\ewaste_vit_project\models\classification\graphs\per_class_f1_comparison.png


## model comparison summary

In [13]:
print("\nmodel comparison summary")
print("=" * 65)
print(f"{'metric':<25} {'resnet18':>12} {'resnet50':>12} {'efficientnet':>12}")
print("-" * 65)

metrics = ["test_accuracy", "macro_f1", "weighted_f1", "macro_precision", "macro_recall"]
for m in metrics:
    vals = [test_results[a][m] for a in ARCHS]
    best = max(vals)
    row  = f"{m:<25}"
    for v in vals:
        marker = " *" if v == best else "  "
        row   += f"{v:>12.4f}{marker}"[:-2] if v == best else f"{v:>12.4f}  "
    print(row)

print("-" * 65)

# identify best model
best_arch = max(ARCHS, key=lambda a: test_results[a]["macro_f1"])
print(f"\nbest model (macro f1): {best_arch}")
print(f"saving best model reference...")

# save best model metadata
with open(OUTPUT_DIR / "best_model.json", "w") as f:
    json.dump({"best_arch": best_arch, "results": json_results[best_arch]}, f, indent=2)
print("done")


model comparison summary
metric                        resnet18     resnet50 efficientnet
-----------------------------------------------------------------
test_accuracy                  0.8575        0.8655      0.8265  
macro_f1                       0.8543        0.8619      0.8207  
weighted_f1                    0.8543        0.8619      0.8207  
macro_precision                0.8569        0.8659      0.8310  
macro_recall                   0.8575        0.8655      0.8265  
-----------------------------------------------------------------

best model (macro f1): resnet50
saving best model reference...
done


## detailed classification report — best model

In [14]:
best_arch = json.load(open(OUTPUT_DIR / "best_model.json"))["best_arch"]
print(f"detailed report for: {best_arch}\n")

report = classification_report(
    test_results[best_arch]["labels"],
    test_results[best_arch]["preds"],
    target_names=CLASS_NAMES,
    digits=4
)
print(report)

# save to text file
with open(OUTPUT_DIR / f"{best_arch}_classification_report.txt", "w") as f:
    f.write(f"model: {best_arch}\n\n")
    f.write(report)
print(f"report saved to {best_arch}_classification_report.txt")

detailed report for: resnet50

                          precision    recall  f1-score   support

                 Battery     0.8167    0.9800    0.8909       100
               Capacitor     0.9070    0.7800    0.8387       100
Integrated-micro-circuit     0.3509    0.4000    0.3738       100
                Keyboard     0.9327    0.9700    0.9510       100
                     LED     0.9118    0.9300    0.9208       100
                  Laptop     0.9596    0.9500    0.9548       100
               Microwave     0.9320    0.9600    0.9458       100
                  Mobile     0.9706    0.9900    0.9802       100
                   Mouse     0.9706    0.9900    0.9802       100
                     PCB     0.7218    0.9600    0.8240       100
                 Printer     0.8796    0.9500    0.9135       100
                Resistor     0.9417    0.9700    0.9557       100
              Television     0.9794    0.9500    0.9645       100
         Washing Machine     0.9898    0.970

## grad-cam — visual explanation of best model predictions

In [15]:
# grad-cam implementation — shows which image regions drive predictions
# critical for research paper interpretability section

from torchvision.transforms.functional import to_pil_image
import torch.nn.functional as F

def get_gradcam_hook(model, arch):
    """register forward/backward hooks on last conv layer."""
    gradients, activations = [], []

    def backward_hook(module, grad_input, grad_output):
        gradients.append(grad_output[0])

    def forward_hook(module, input, output):
        activations.append(output)

    if arch in ["resnet18", "resnet50"]:
        target_layer = model.layer4[-1].conv2
    elif arch == "efficientnet_b0":
        target_layer = model.features[-1][0]

    fh = target_layer.register_forward_hook(forward_hook)
    bh = target_layer.register_full_backward_hook(backward_hook)

    return gradients, activations, fh, bh


def compute_gradcam(model, image_tensor, label, arch):
    model.eval()
    gradients, activations, fh, bh = get_gradcam_hook(model, arch)

    image_tensor = image_tensor.unsqueeze(0).to(device)
    output = model(image_tensor)
    model.zero_grad()
    output[0, label].backward()

    grad  = gradients[0].cpu().detach()
    activ = activations[0].cpu().detach()

    weights = grad.mean(dim=(2, 3), keepdim=True)
    cam     = (weights * activ).sum(dim=1, keepdim=True)
    cam     = F.relu(cam)
    cam     = F.interpolate(cam, size=(224, 224), mode="bilinear", align_corners=False)
    cam     = cam.squeeze().numpy()
    cam     = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)

    fh.remove()
    bh.remove()
    return cam


# generate grad-cam for one sample per class
best_model = all_models[best_arch]
best_model.eval()

fig, axes = plt.subplots(4, 5, figsize=(20, 16))
fig.suptitle(f"grad-cam visualizations — {best_arch}", fontsize=13, fontweight="bold")

inv_normalize = transforms.Normalize(
    mean=[-m/s for m, s in zip(MEAN, STD)],
    std=[1/s for s in STD]
)

for cls_idx, cls_name in enumerate(CLASS_NAMES):
    ax = axes[cls_idx // 5, cls_idx % 5]
    cls_samples = [s for s in test_ds.samples if s[1] == cls_idx]
    if not cls_samples:
        ax.axis("off")
        continue

    img_path, label = cls_samples[0]
    img_tensor = eval_transform(
        __import__("PIL").Image.open(img_path).convert("RGB")
    )

    cam = compute_gradcam(best_model, img_tensor, label, best_arch)

    # original image
    orig = inv_normalize(img_tensor).permute(1, 2, 0).numpy()
    orig = np.clip(orig, 0, 1)

    # overlay
    heatmap = plt.cm.jet(cam)[:, :, :3]
    overlay = 0.5 * orig + 0.5 * heatmap

    ax.imshow(overlay)
    ax.set_title(cls_name, fontsize=8, pad=2)
    ax.axis("off")

plt.tight_layout()
save_path = GRAPHS_DIR / "gradcam_all_classes.png"
plt.savefig(save_path, dpi=150, bbox_inches="tight")
plt.close()
print(f"saved: {save_path}")
torch.cuda.empty_cache()

saved: D:\Github Desktop\ewaste_vit_project\models\classification\graphs\gradcam_all_classes.png


## classification complete
Outputs saved:
- models: `models/classification/{arch}/{arch}_best.pth`
- graphs: `models/classification/graphs/`
- results: `models/classification/test_results.json`
- report: `models/classification/{best_arch}_classification_report.txt`